# 01 - Data Collection

Notebook aligned with the project structure defined in `README.md`.

Scope of this notebook:
- define the multi-asset universe
- inspect the raw ingestion output
- document the collection steps you are following
- keep local raw files in CSV/JSON and S3 objects in Parquet


## Project Parameters

These defaults follow the current project configuration:
- symbols: `NVDA`, `AMD`, `TSM`, `ASML`, `QCOM`
- historical window: `2018-01-01` to `2025-12-31`
- raw local zone: `data/raw`


In [ ]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_ROOT = PROJECT_ROOT / "data" / "raw"

SYMBOLS = ["NVDA", "AMD", "TSM", "ASML", "QCOM"]
START_DATE = "2018-01-01"
END_DATE = "2025-12-31"
EXTRACTION_DATE = "2026-04-19"

RAW_ROOT

## Raw Manifest

If you have already run `python scripts/generate_raw.py --skip-s3` or `python scripts/generate_raw.py`, use the cells below to inspect the generated raw manifest.

In [ ]:
manifest_path = RAW_ROOT / "manifests" / f"extraction_date={EXTRACTION_DATE}" / "raw_manifest.json"
manifest_path

In [ ]:
with manifest_path.open("r", encoding="utf-8") as fp:
    raw_manifest = json.load(fp)

raw_manifest

In [ ]:
manifest_assets = pd.DataFrame(raw_manifest["assets"])
manifest_assets

## Sample Asset Inspection

Use one asset first to validate the schema and date coverage.

In [ ]:
sample_symbol = "AMD"
sample_path = RAW_ROOT / "market_data" / "source=yfinance" / f"symbol={sample_symbol}" / f"extraction_date={EXTRACTION_DATE}" / "ohlcv.csv"
sample_path

In [ ]:
sample_df = pd.read_csv(sample_path, parse_dates=["date"])
sample_df.head()

In [ ]:
sample_df.info()

In [ ]:
sample_df[["date", "close"]].set_index("date").plot(figsize=(12, 4), title=f"{sample_symbol} close price")

## Multi-Asset Loading

This cell follows the same project universe described in the README.

In [ ]:
assets = {}

for symbol in SYMBOLS:
    path = RAW_ROOT / "market_data" / "source=yfinance" / f"symbol={symbol}" / f"extraction_date={EXTRACTION_DATE}" / "ohlcv.csv"
    df = pd.read_csv(path, parse_dates=["date"])
    assets[symbol] = df[["date", "close"]].dropna().copy()

{symbol: frame.shape for symbol, frame in assets.items()}

## Notes And Manual Steps

Use this section to record the exact collection workflow you are following.

Suggested checklist:
1. confirm environment variables in `.env`
2. run the raw generation script
3. validate local manifest and CSV schema
4. validate the S3 parquet layout when the bucket is configured
5. record any deviations from the default symbol/date scope


In [ ]:
# Add your own step-by-step notes here.
collection_notes = {
    "status": "draft",
    "owner": "",
    "steps": []
}

collection_notes